# Lane Detection with PyTorch - Video Support Edition

This notebook provides a complete lane detection system with **video dataset support**.

**Features:**
- Image-based training (original)
- **Video sequence training** with temporal information
- Temporal frame stacking for better continuity
- Real-time video and webcam inference

**Run all cells sequentially for a complete demo.**

## 0. Configuration - Choose Mode

In [ ]:
# Configuration flags
VIDEO_MODE = True  # Set to True for video sequence training, False for image-only
SEQUENCE_LENGTH = 5  # Number of frames in sequence (for video mode)
EPOCHS = 10  # Training epochs
BATCH_SIZE = 4  # Batch size (reduce if OOM)

print(f"Configuration:")
print(f"  VIDEO_MODE: {VIDEO_MODE}")
print(f"  SEQUENCE_LENGTH: {SEQUENCE_LENGTH}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  BATCH_SIZE: {BATCH_SIZE}")

## 1. Setup and Imports

In [ ]:
import os
import sys
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, 'src')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 2. Video Dataset Preparation

Generate synthetic video clips with lane labels (TuSimple-like format)

In [ ]:
from video_dataset_generator import VideoLaneDatasetGenerator

# Generate video dataset
video_generator = VideoLaneDatasetGenerator(
    data_dir="data/tusimple_video",
    num_clips=50,
    frames_per_clip=20
)
video_generator.generate_dataset()

print("\nVideo dataset ready!")

## 3. Dataset Classes

In [ ]:
from video_dataset import VideoLaneDataset, TemporalLaneModel

# Transforms
video_train_transform = A.Compose([
    A.Resize(256, 384),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=5, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

video_val_transform = A.Compose([
    A.Resize(256, 384),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

if VIDEO_MODE:
    # Video dataset
    train_dataset = VideoLaneDataset(
        "data/tusimple_video",
        sequence_length=SEQUENCE_LENGTH,
        transform=video_train_transform,
        split='train'
    )
    val_dataset = VideoLaneDataset(
        "data/tusimple_video",
        sequence_length=SEQUENCE_LENGTH,
        transform=video_val_transform,
        split='val'
    )
else:
    # Fallback to image dataset
    print("Using image mode - falling back to synthetic image dataset")
    from dataset_handler import LaneDatasetDownloader
    downloader = LaneDatasetDownloader()
    downloader.prepare_dataset(num_synthetic=200)
    
    # Import image dataset class
    from train import LaneDataset
    train_dataset = LaneDataset("data/train/images", "data/train/masks", transform=video_train_transform)
    val_dataset = LaneDataset("data/val/images", "data/val/masks", transform=video_val_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nTrain: {len(train_dataset)}, Val: {len(val_dataset)}")

## 4. Visualize Video Samples

In [ ]:
def visualize_video_sample(dataset, idx=0):
    """Visualize a video sequence sample"""
    frames, mask = dataset[idx]
    
    # frames: [T, C, H, W]
    T = frames.shape[0]
    
    fig, axes = plt.subplots(2, T, figsize=(T*3, 6))
    
    for t in range(T):
        # Denormalize frame
        frame = frames[t].permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        frame = std * frame + mean
        frame = np.clip(frame, 0, 1)
        
        axes[0, t].imshow(frame)
        axes[0, t].set_title(f'Frame {t+1}')
        axes[0, t].axis('off')
    
    # Show mask on last frame
    mask_np = mask.numpy() if isinstance(mask, torch.Tensor) else mask
    for t in range(T):
        axes[1, t].imshow(mask_np, cmap='gray')
        axes[1, t].set_title(f'Mask (Frame {T})' if t == T-1 else '')
        axes[1, t].axis('off')
    
    plt.tight_layout()
    plt.savefig('video_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved video_sample.png")

if VIDEO_MODE and len(train_dataset) > 0:
    visualize_video_sample(train_dataset, idx=0)

## 5. Model with Temporal Support

In [ ]:
class LaneDetectionModel(nn.Module):
    def __init__(self, encoder_name="resnet18", encoder_weights="imagenet", video_mode=False, sequence_length=5):
        super().__init__()
        self.video_mode = video_mode
        self.sequence_length = sequence_length
        
        # Base U-Net model
        self.base_model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3 * sequence_length if video_mode else 3,
            classes=1,
            activation=None
        )
    
    def forward(self, x):
        """
        Args:
            x: If video_mode: [B, T, C, H, W], else: [B, C, H, W]
        """
        if self.video_mode and x.dim() == 5:
            # Stack temporal frames as channels: [B, T, C, H, W] -> [B, T*C, H, W]
            B, T, C, H, W = x.shape
            x = x.view(B, T * C, H, W)
        
        return self.base_model(x)

# Initialize model
model = LaneDetectionModel(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    video_mode=VIDEO_MODE,
    sequence_length=SEQUENCE_LENGTH
).to(DEVICE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Video mode: {VIDEO_MODE}")

## 6. Loss and Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
    
    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + self.dice_weight * self.dice(pred, target)

def calculate_iou(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    target = target.float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return (intersection / union).item()

criterion = CombinedLoss()
print("Loss ready")

## 7. Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, video_mode):
    model.train()
    total_loss, total_iou = 0, 0
    
    for data, masks in tqdm(loader, desc="Training"):
        data, masks = data.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_iou += calculate_iou(outputs, masks)
    
    return total_loss / len(loader), total_iou / len(loader)

def validate_epoch(model, loader, criterion, device, video_mode):
    model.eval()
    total_loss, total_iou = 0, 0
    
    with torch.no_grad():
        for data, masks in tqdm(loader, desc="Validation"):
            data, masks = data.to(device), masks.to(device)
            outputs = model(data)
            loss = criterion(outputs, masks)
            
            total_loss += loss.item()
            total_iou += calculate_iou(outputs, masks)
    
    return total_loss / len(loader), total_iou / len(loader)

# Setup
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
history = {'train_loss': [], 'train_iou': [], 'val_loss': [], 'val_iou': []}
best_iou = 0

print(f"\nStarting training for {EPOCHS} epochs...")

## 8. Training Loop

In [ ]:
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 50)
    
    train_loss, train_iou = train_epoch(model, train_loader, optimizer, criterion, DEVICE, VIDEO_MODE)
    val_loss, val_iou = validate_epoch(model, val_loader, criterion, DEVICE, VIDEO_MODE)
    
    scheduler.step(val_iou)
    
    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    
    print(f"Train - Loss: {train_loss:.4f}, IoU: {train_iou:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, IoU: {val_iou:.4f}")
    
    if val_iou > best_iou:
        best_iou = val_iou
        model_name = 'best_video_model.pth' if VIDEO_MODE else 'best_image_model.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_iou': best_iou,
            'video_mode': VIDEO_MODE,
            'sequence_length': SEQUENCE_LENGTH if VIDEO_MODE else None
        }, f'models/{model_name}')
        print(f"✓ Saved best model (IoU: {best_iou:.4f})")

print(f"\nTraining complete! Best Val IoU: {best_iou:.4f}")

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-', label='Train Loss')
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_iou'], 'b-', label='Train IoU')
axes[1].plot(epochs, history['val_iou'], 'r-', label='Val IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].set_title('Training & Validation IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('video_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved video_training_curves.png")

## 10. Video Inference Functions

In [ ]:
class VideoLaneDetector:
    """Real-time video lane detection with smoothing"""
    
    def __init__(self, model, device, sequence_length=5, buffer_size=3):
        self.model = model
        self.device = device
        self.sequence_length = sequence_length
        self.frame_buffer = []
        self.buffer_size = buffer_size
        
        # For smoothing
        self.prev_lanes = None
        self.smooth_factor = 0.7
    
    def preprocess_frame(self, frame):
        """Preprocess single frame"""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (384, 256))
        frame_norm = frame_resized / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        frame_norm = (frame_norm - mean) / std
        return torch.from_numpy(frame_norm.transpose(2, 0, 1)).float()
    
    def get_sequence(self, current_frame):
        """Get sequence of frames for video model"""
        self.frame_buffer.append(current_frame)
        if len(self.frame_buffer) > self.sequence_length:
            self.frame_buffer.pop(0)
        
        # Pad if needed
        while len(self.frame_buffer) < self.sequence_length:
            self.frame_buffer.insert(0, self.frame_buffer[0])
        
        return torch.stack(self.frame_buffer)
    
    def detect_lanes(self, frame, mask):
        """Detect and smooth lane lines from mask"""
        h, w = frame.shape[:2]
        mask_resized = cv2.resize(mask, (w, h))
        mask_binary = (mask_resized > 0.5).astype(np.uint8)
        
        # Find contours
        contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        lanes = {'left': None, 'right': None}
        
        if len(contours) >= 1:
            all_points = []
            for cnt in contours:
                if len(cnt) >= 5:
                    for point in cnt:
                        all_points.append(point[0])
            
            if len(all_points) >= 10:
                all_points = np.array(all_points)
                center_x = w // 2
                left_points = all_points[all_points[:, 0] < center_x]
                right_points = all_points[all_points[:, 0] >= center_x]
                
                if len(left_points) >= 5:
                    lanes['left'] = np.polyfit(left_points[:, 1], left_points[:, 0], 2)
                if len(right_points) >= 5:
                    lanes['right'] = np.polyfit(right_points[:, 1], right_points[:, 0], 2)
        
        # Temporal smoothing
        if self.prev_lanes is not None:
            for side in ['left', 'right']:
                if lanes[side] is not None and self.prev_lanes[side] is not None:
                    lanes[side] = (self.smooth_factor * self.prev_lanes[side] + 
                                   (1 - self.smooth_factor) * lanes[side])
        
        self.prev_lanes = lanes
        return lanes
    
    def draw_lanes(self, frame, lanes, color_left=(0, 255, 0), color_right=(0, 0, 255)):
        """Draw fitted lanes on frame"""
        result = frame.copy()
        h, w = frame.shape[:2]
        y_vals = np.linspace(h//2, h-1, 50, dtype=int)
        
        if lanes['left'] is not None:
            left_x = np.polyval(lanes['left'], y_vals).astype(int)
            left_points = np.column_stack((left_x, y_vals))
            left_points = left_points[(left_points[:, 0] >= 0) & (left_points[:, 0] < w)]
            if len(left_points) > 1:
                cv2.polylines(result, [left_points], False, color_left, 6)
        
        if lanes['right'] is not None:
            right_x = np.polyval(lanes['right'], y_vals).astype(int)
            right_points = np.column_stack((right_x, y_vals))
            right_points = right_points[(right_points[:, 0] >= 0) & (right_points[:, 0] < w)]
            if len(right_points) > 1:
                cv2.polylines(result, [right_points], False, color_right, 6)
        
        # Fill lane area
        if lanes['left'] is not None and lanes['right'] is not None:
            left_x = np.polyval(lanes['left'], y_vals).astype(int)
            right_x = np.polyval(lanes['right'], y_vals).astype(int)
            pts_left = np.column_stack((left_x, y_vals))
            pts_right = np.column_stack((right_x, y_vals))[::-1]
            pts = np.vstack((pts_left, pts_right))
            pts = pts[(pts[:, 0] >= 0) & (pts[:, 0] < w)]
            
            if len(pts) >= 3:
                overlay = result.copy()
                cv2.fillPoly(overlay, [pts], (0, 200, 0))
                result = cv2.addWeighted(result, 0.8, overlay, 0.2, 0)
        
        return result
    
    def process_frame(self, frame):
        """Process single frame and return result"""
        h, w = frame.shape[:2]
        
        # Preprocess
        frame_tensor = self.preprocess_frame(frame)
        
        if VIDEO_MODE:
            # Get sequence
            sequence = self.get_sequence(frame_tensor)
            input_tensor = sequence.unsqueeze(0).to(self.device)
        else:
            input_tensor = frame_tensor.unsqueeze(0).to(self.device)
        
        # Predict
        self.model.eval()
        with torch.no_grad():
            pred = self.model(input_tensor)
            pred_mask = torch.sigmoid(pred).squeeze().cpu().numpy()
        
        # Detect and draw lanes
        lanes = self.detect_lanes(frame, pred_mask)
        result = self.draw_lanes(frame, lanes)
        
        return result, pred_mask

print("VideoLaneDetector ready")

## 11. Process Video File

In [ ]:
def process_video_file(model, input_path, output_path, device, max_frames=None):
    """Process video file with lane detection"""
    cap = cv2.VideoCapture(str(input_path))
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
    
    # Initialize detector
    detector = VideoLaneDetector(model, device, sequence_length=SEQUENCE_LENGTH if VIDEO_MODE else 1)
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)
    
    frame_count = 0
    
    with tqdm(total=total_frames, desc="Processing video") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret or (max_frames and frame_count >= max_frames):
                break
            
            result, _ = detector.process_frame(frame)
            out.write(result)
            
            frame_count += 1
            pbar.update(1)
    
    cap.release()
    out.release()
    print(f"\nVideo saved to {output_path}")
    print(f"Processed {frame_count} frames")

# Load best model for inference
model_name = 'best_video_model.pth' if VIDEO_MODE else 'best_image_model.pth'
checkpoint = torch.load(f'models/{model_name}', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']+1} (IoU: {checkpoint['best_iou']:.4f})")

# Process a sample clip from our dataset
sample_clip = list(Path("data/tusimple_video/clips").glob("clip_*"))[0]
print(f"\nProcessing sample clip: {sample_clip.name}")

# Create a simple video from the clip frames
frames = sorted([f for f in sample_clip.glob("*.jpg")], key=lambda x: int(x.stem))
print(f"Found {len(frames)} frames")

# Write sample input video
input_video = "sample_input.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(input_video, fourcc, 10, (1280, 720))
for frame_path in frames:
    frame = cv2.imread(str(frame_path))
    out.write(frame)
out.release()
print(f"Created input video: {input_video}")

# Process with lane detection
process_video_file(model, input_video, "video_output.mp4", DEVICE, max_frames=len(frames))

## 12. Webcam Inference (Optional)

Run real-time lane detection on webcam. Press 'q' to quit.

In [ ]:
def webcam_inference(model, device, camera_index=0):
    """Real-time webcam lane detection"""
    cap = cv2.VideoCapture(camera_index)
    
    if not cap.isOpened():
        print(f"Cannot open camera {camera_index}")
        return
    
    # Initialize detector
    detector = VideoLaneDetector(model, device, sequence_length=SEQUENCE_LENGTH if VIDEO_MODE else 1)
    
    print("\nWebcam started. Press 'q' to quit.")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        result, _ = detector.process_frame(frame)
        
        # Display
        cv2.imshow('Lane Detection', result)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam stopped.")

# Uncomment to run webcam inference
# webcam_inference(model, DEVICE, camera_index=0)

## 13. Summary and Results

In [ ]:
print("=" * 60)
print("VIDEO LANE DETECTION PROJECT COMPLETE")
print("=" * 60)
print(f"\nMode: {'VIDEO' if VIDEO_MODE else 'IMAGE'}")
if VIDEO_MODE:
    print(f"Sequence Length: {SEQUENCE_LENGTH}")
print(f"Best Validation IoU: {best_iou:.4f}")
print(f"Device: {DEVICE}")

print("\nGenerated files:")
files = [
    'video_sample.png',
    'video_training_curves.png',
    'sample_input.mp4',
    'video_output.mp4',
    f'models/{model_name}'
]
for f in files:
    if os.path.exists(f):
        print(f"  ✓ {f}")

print("\nTo run on your own video:")
print("  process_video_file(model, 'input.mp4', 'output.mp4', DEVICE)")
print("\nTo run on webcam:")
print("  webcam_inference(model, DEVICE)")
print("=" * 60)